In [ ]:
import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

In [ ]:
#!pip install -q pmdarima
#!pip install -q tbats
!pip install -q sktime

In [ ]:
import sys
import logging
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from tqdm import tqdm
from statsmodels.tsa.stattools import adfuller, kpss
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
from sktime.utils.plotting import plot_series
from sktime.forecasting.base import ForecastingHorizon
from sktime.forecasting.exp_smoothing import ExponentialSmoothing
from sktime.forecasting.sarimax import SARIMAX
#from sktime.forecasting.arima import ARIMA, AutoARIMA
#from sktime.forecasting.bats import BATS
#from sktime.forecasting.tbats import TBATS
#from sktime.forecasting.ets import AutoETS
#from sktime.forecasting.compose import make_reduction
#from sklearn.linear_model import LinearRegression
#from xgboost import XGBRegressor

from statsmodels.tools.sm_exceptions import ConvergenceWarning
warnings.simplefilter('ignore', ConvergenceWarning)
warnings.simplefilter("ignore", RuntimeWarning)
pd.options.mode.chained_assignment = None

In [ ]:
project_path = "/kaggle/input/store-sales-time-series-forecasting/"

df_stores = pd.read_csv(project_path + "stores.csv")
df_oil = pd.read_csv(project_path + "oil.csv", parse_dates=["date"])
df_holidays = pd.read_csv(project_path + "holidays_events.csv", parse_dates=["date"])
df_transactions = pd.read_csv(project_path + "transactions.csv", parse_dates=["date"])

df = pd.read_csv(project_path + "train.csv", parse_dates=["date"])
df_test = pd.read_csv(project_path + "test.csv", parse_dates=["date"])

In [ ]:
FIRST_DATE = df["date"].min()
LAST_DATE = df_test["date"].max()
VAL_DATE = pd.to_datetime("2017-08-01")
ALL_DATES = pd.date_range(FIRST_DATE, LAST_DATE, freq='D')
CHRISTMAS_DAYS = [pd.to_datetime(f"201{n}-12-25") for n in [3,4,5,6]]
N_STORES = df["store_nbr"].nunique()
N_FAMILIES = df["family"].nunique()

VALIDATION = False
FIT_RESIDUALS = False

In [ ]:
def plot_predictions(y_train, y_pred, y_fit=None, y_val=None):
    
    plot_params = dict(
        color='0.75',
        style='.-',
        markeredgecolor='0.25',
        markerfacecolor='0.25'
    )
    ax = y_train.plot(**plot_params, figsize=(10, 4))
    ax = y_pred.plot(ax=ax, color='tab:orange', label="forecast")
    if y_fit is not None:
        ax = y_fit.plot(ax=ax, color='tab:blue', label="fit")
    if y_val is not None:
        ax = y_val.plot(ax=ax, style='k.-')
        ax.set_xlim(pd.to_datetime("2017-05-01"), pd.to_datetime("2017-08-15"))
    else:
        ax.set_xlim(pd.to_datetime("2017-05-01"), pd.to_datetime("2017-08-31"))
    ax.set_title("Data and Predictions"); ax.set_xlabel("date"); ax.legend();

In [ ]:
def RMSLE(y_true, y_pred):
    
    if isinstance(y_true, pd.Series):
        y_true = y_true.to_numpy()
    if isinstance(y_pred, pd.Series):
        y_pred = y_pred.to_numpy()
        
    return np.sqrt(np.mean((np.log(1 + y_pred) - np.log(1 + y_true))**2))

In [ ]:
df_stores.head(10)

In [ ]:
df_oil = df_oil.rename({"dcoilwtico": "oil_price"}, axis=1)
df_oil = df_oil.set_index("date").reindex(ALL_DATES).reset_index(names="date")
df_oil["oil_price"] = df_oil["oil_price"].interpolate(limit_direction="both")

df_oil

In [ ]:
df_oil.plot(x="date", y="oil_price", ylabel="oil price",
            title="Oil price over time", figsize=(15, 10));

In [ ]:
df_transactions

In [ ]:
df_holidays.head(10)

In [ ]:
df.head(10)

In [ ]:
df[df["date"].isin(CHRISTMAS_DAYS)]

In [ ]:
# Set date frequency to day to include all days (ie christmas days)

df = df.set_index(["date", "store_nbr", "family"])\
        .unstack(["store_nbr", "family"]).asfreq('D')\
        .stack(["store_nbr", "family"], future_stack=True)\
        .reset_index()

# Fill in missing christmas days

df["sales"] = df["sales"].fillna(0.0)
df["onpromotion"] = df["onpromotion"].fillna(0).astype(int)

df[df["date"].isin(CHRISTMAS_DAYS)]

In [ ]:
df = pd.merge(df, df_oil, on="date", how="left")

df

In [ ]:
df_test = pd.merge(df_test, df_oil, on="date", how="left")

df_test

In [ ]:
df_sales_per_store = df.groupby(["date", "store_nbr"])["sales"].sum().reset_index()\
.pivot(index="date", columns="store_nbr", values="sales")

df_sales_per_store

In [ ]:
df_sales_per_store[range(1,11)].resample('W').mean()\
.plot(figsize=(15,10), ylabel="sales", title= "Sales per store");

In [ ]:
df_sales_per_family = df.groupby(["date", "family"])["sales"].sum().reset_index()\
.pivot(index="date", columns="family", values="sales")

df_sales_per_family

In [ ]:
df_sales_per_family.resample('W').mean()\
.plot(figsize=(15,10), ylabel="sales", title= "Sales per family");

In [ ]:
store_nbr = 38 #np.random.randint(N_STORES)
family = "LIQUOR,WINE,BEER" #df["family"].unique()[np.random.randint(N_FAMILIES)]

df_store_family = df[(df["store_nbr"]==store_nbr) & (df["family"]==family)]

df_store_family

In [ ]:
df_store_family.set_index("date")["sales"].resample('W').mean()\
.plot(figsize=(12, 7), ylabel="sales", 
      title=f"Sales for store {store_nbr} and family {family}");

In [ ]:
df_train = df_store_family[df_store_family["date"] < VAL_DATE].set_index("date").asfreq('D')
df_val = df_store_family[df_store_family["date"] >= VAL_DATE].set_index("date").asfreq('D')

X_train = df_train[["onpromotion", "oil_price"]]
y_train = df_train["sales"]

X_val = df_val[["onpromotion", "oil_price"]]
y_val = df_val["sales"]

y_train.loc[CHRISTMAS_DAYS]

In [ ]:
model = ExponentialSmoothing(trend='additive', seasonal='additive', sp=7)
%time model.fit(y_train)

In [ ]:
fh_train = ForecastingHorizon(y_train.index, freq='D')
y_fit = model.predict(fh_train).clip(lower=0)

fh_val = ForecastingHorizon(y_val.index, freq='D')
y_pred = model.predict(fh_val).clip(lower=0)

y_pred

In [ ]:
if FIT_RESIDUALS:
    
    y_residual = y_train - y_fit
    model_residual = SARIMAX(order=(2,0,0))
    %time model_residual.fit(y_residual, X=X_train)

In [ ]:
if FIT_RESIDUALS:

    y_residual_fit = model_residual.predict(fh_train, X=X_train)
    y_residual_pred = model_residual.predict(fh_val, X=X_val)
    
    y_fit = (y_fit + y_residual_fit).clip(lower=0)
    y_pred = (y_pred + y_residual_pred).clip(lower=0)
    
    display(y_pred)

In [ ]:
plot_predictions(y_train, y_pred, y_fit, y_val)

In [ ]:
RMSLE(y_val, y_pred)

In [ ]:
if VALIDATION:

    df_train = df[df["date"] < VAL_DATE].set_index(["store_nbr", "family", "date"]).sort_index()
    df_val = df[df["date"] >= VAL_DATE].set_index(["store_nbr", "family", "date"]).sort_index()

    X_train = df_train[["onpromotion", "oil_price"]]
    y_train = df_train[["sales"]]
        
    X_val = df_val[["onpromotion", "oil_price"]]
    y_val = df_val[["sales"]]
    
    display(y_train)

In [ ]:
if VALIDATION:

    model = ExponentialSmoothing(trend='additive', seasonal='additive', sp=7)
    %time model.fit(y_train)

In [ ]:
if VALIDATION:

    train_dates = y_train.index.levels[-1]
    fh_train= ForecastingHorizon(train_dates, freq='D')
    y_fit = model.predict(fh_train).clip(lower=0)
    
    validation_dates = y_val.index.levels[-1]
    fh_val = ForecastingHorizon(validation_dates, freq='D')
    y_pred = model.predict(fh_val).clip(lower=0)

    display(y_pred)

In [ ]:
if VALIDATION and FIT_RESIDUALS:

    y_residual = y_train - y_fit
    model_residual = SARIMAX(order=(2,0,0))
    %time model_residual.fit(y_residual, X=X_train)

In [ ]:
if VALIDATION and FIT_RESIDUALS:

    y_residual_fit = model_residual.predict(fh_train, X=X_train)
    y_residual_pred = model_residual.predict(fh_val, X=X_val)
    
    y_fit = (y_fit + y_residual_fit).clip(lower=0)
    y_pred = (y_pred + y_residual_pred).clip(lower=0)
    
    display(y_pred)

In [ ]:
if VALIDATION:

    plot_predictions(y_train.loc[store_nbr, family], 
                     y_pred.loc[store_nbr, family], 
                     y_fit.loc[store_nbr, family], 
                     y_val.loc[store_nbr, family])

In [ ]:
if VALIDATION:

    df_results = pd.merge(df_val.reset_index(), y_pred.reset_index(), 
                          on=["date", "store_nbr", "family"], how="left", 
                          suffixes=("_true", "_pred")).sort_values("id")\
                        .reset_index(drop=True)
    
    display(df_results)

In [ ]:
if VALIDATION:

    print(RMSLE(df_results["sales_true"], df_results["sales_pred"]))

In [ ]:
df_train = df.set_index(["store_nbr", "family", "date"]).sort_index()
X_test = df_test.set_index(["store_nbr", "family", "date"]).sort_index()

X_test = X_test[["onpromotion", "oil_price"]]
X_train = df_train[["onpromotion", "oil_price"]]
y_train = df_train[["sales"]]

y_train

In [ ]:
model = ExponentialSmoothing(trend='additive', seasonal='additive', sp=7)
%time model.fit(y_train)

In [ ]:
train_dates = y_train.index.levels[-1]
fh_train = ForecastingHorizon(train_dates, freq='D')
y_fit = model.predict(fh_train).clip(lower=0)

test_dates = X_test.index.levels[-1]
fh_test = ForecastingHorizon(test_dates, freq='D')
y_pred = model.predict(fh_test).clip(lower=0)

y_pred

In [ ]:
if FIT_RESIDUALS:

    y_residual = y_train - y_fit
    model_residual = SARIMAX(order=(2,0,0))
    %time model_residual.fit(y_residual, X=X_train)

In [ ]:
if FIT_RESIDUALS:

    y_residual_fit = model_residual.predict(fh_train, X=X_train)
    y_residual_pred = model_residual.predict(fh_test, X=X_test)
    
    y_fit = (y_fit + y_residual_fit).clip(lower=0)
    y_pred = (y_pred + y_residual_pred).clip(lower=0)
    
    display(y_pred)

In [ ]:
plot_predictions(y_train.loc[store_nbr, family].squeeze(), 
                 y_pred.loc[store_nbr, family].squeeze(), 
                 y_fit.loc[store_nbr, family].squeeze())

In [ ]:
df_results = pd.merge(df_test, y_pred.reset_index(), 
                      on=["date", "store_nbr", "family"], 
                      how="left").sort_values("id")

df_results

In [ ]:
df_submission = pd.read_csv(project_path + "sample_submission.csv")

df_submission["sales"] = df_results["sales"]

df_submission

In [ ]:
df_submission.to_csv("submission.csv", index=False)